# Image Model Deployment to Databricks Model Serving

This notebook deploys a registered image model to Databricks Model Serving on GPU_LARGE instances.

## Install Required Packages

In [0]:
%pip install mlflow[databricks] databricks-sdk --upgrade
dbutils.library.restartPython()

## Setup Widgets for Configuration

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "default", "Schema Name")
dbutils.widgets.text("model_name", "image_model", "Model Name")
dbutils.widgets.text("endpoint_name", "image_model_endpoint", "Endpoint Name")
dbutils.widgets.text("model_alias", "staging", "Model stage")

## Read Configuration from Widgets

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = dbutils.widgets.get("model_name")
endpoint_name = dbutils.widgets.get("endpoint_name")
model_stage = dbutils.widgets.get("model_alias")

registered_model_name = f"{catalog}.{schema}.{model_name}"

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Registered Model: {registered_model_name}")
print(f"Model Stage: {model_stage}")
print(f"Endpoint Name: {endpoint_name}")

## Initialize Databricks Workspace Client

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AutoCaptureConfigInput,
    ServingModelWorkloadType
)

w = WorkspaceClient()
print(f"Workspace URL: {w.config.host}")

model_version = w.model_versions.get_by_alias(registered_model_name, model_stage).version
model_version

## Deploy Model to Serving Endpoint

This will create or update a model serving endpoint with GPU_LARGE workload size.

In [0]:
# Check if endpoint exists
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists. Updating...")
    endpoint_exists = True
except Exception as e:
    print(f"Endpoint '{endpoint_name}' does not exist. Creating new endpoint...")
    endpoint_exists = False

In [0]:
# Configure the served entity
served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_size="Small",
    workload_type=ServingModelWorkloadType.GPU_LARGE,
    scale_to_zero_enabled=True,
)

if endpoint_exists:
    # Update existing endpoint
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=[served_entity]
    )
    print(f"Endpoint '{endpoint_name}' updated successfully!")
else:
    # Create new endpoint
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            name=endpoint_name,
            served_entities=[served_entity]
        )
    )
    print(f"Endpoint '{endpoint_name}' created successfully!")

## Get Endpoint Details

In [0]:
endpoint = w.serving_endpoints.get(endpoint_name)
print(f"\nEndpoint Name: {endpoint.name}")
print(f"Endpoint State: {endpoint.state.ready}")
print(f"Endpoint URL: {w.config.host}/serving-endpoints/{endpoint_name}/invocations")